# Image-Generation Prompt Optimization Pipeline

End-to-end run of the multimodal Promptomatix pipeline on the RCP cluster.

**This notebook is intended to be executed on the server pod only** (it expects `/scratch/Mock_repo_mnlp` to exist and a GPU to be available for the local vLLM + Stable Diffusion servers).

Pipeline summary:
1. Install the necessary Python packages (Promptomatix + multimodal extras).
2. Launch the local vLLM (`Qwen2-VL-7B-Instruct`) and Stable Diffusion (`SDXL base 1.0`) servers and run the optimizer through `scratch/run_with_servers.sh`.
3. Display the images that the diffusion server produced during optimization as proof that the pipeline ran end-to-end.

## 1. Install dependencies

Install Promptomatix (editable, from this repo) together with `vllm`, `diffusers`, `transformers`, `accelerate`, and `flask` (used by `scratch/local_diffusion_server.py`).

In [ ]:
%cd /scratch/Mock_repo_mnlp

# `blinker` is preinstalled in the base image via distutils, which prevents pip
# from upgrading it when Flask (a Promptomatix transitive dep) pulls a newer
# version. `--ignore-installed blinker` skips the uninstall step and avoids the
# `uninstall-distutils-installed-package` error.
!pip install -q --ignore-installed blinker

# Promptomatix (editable install -- picks up local changes to promptomatix/src/)
!pip install -q -e promptomatix

# Multimodal serving stack used by scratch/run_with_servers.sh
!pip install -q \
    "vllm>=0.5.0" \
    "diffusers>=0.25.0" \
    "transformers>=4.38.0" \
    "accelerate>=0.27.0" \
    "torch>=2.2.0" \
    "flask>=2.0.0" \
    "litellm"

# Quick sanity check
import torch, sys
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu 0  :", torch.cuda.get_device_name(0))

## 2. Run the pipeline

`scratch/run_with_servers.sh`:
1. Spawns the local vLLM VLM server (`Qwen/Qwen2-VL-7B-Instruct`) on port 8000.
2. Spawns the local Stable Diffusion server (`SDXL base 1.0`, from `scratch/local_diffusion_server.py`) on port 8001.
3. Waits for both ports to be online.
4. Runs the Promptomatix optimizer (`simple_meta_prompt` backend) for the `image_generation` task on three seed concepts.
5. Tears down the background servers via the script's `trap cleanup EXIT`.

Output is `tee`'d to `/scratch/Mock_repo_mnlp/outputs/pipeline.log` so we keep a copy on the PVC.

In [ ]:
%cd /scratch/Mock_repo_mnlp
!mkdir -p /scratch/Mock_repo_mnlp/outputs/generated_images
!bash scratch/run_with_servers.sh 2>&1 | tee /scratch/Mock_repo_mnlp/outputs/pipeline.log

## 3. Proof: generated images

List every image the diffusion server saved under `/scratch/Mock_repo_mnlp/outputs/generated_images/` during the optimizer run, then render them inline.

In [ ]:
from pathlib import Path
import random
from IPython.display import Image, display, Markdown

IMG_ROOT = Path("/scratch/Mock_repo_mnlp/outputs/generated_images")
PHASES = ("synthetic_data", "final_optimized")
N_PER_PHASE = 3
SEED = None  # set to an int for reproducible sampling

if SEED is not None:
    random.seed(SEED)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

if not IMG_ROOT.exists():
    print(f"No image directory found at {IMG_ROOT}")
else:
    concept_dirs = sorted(p for p in IMG_ROOT.iterdir() if p.is_dir())
    if not concept_dirs:
        print(f"No per-concept folders found under {IMG_ROOT}")
    for concept_dir in concept_dirs:
        display(Markdown(f"## `{concept_dir.name}`"))
        for phase in PHASES:
            phase_dir = concept_dir / phase
            if not phase_dir.exists():
                display(Markdown(f"_(no `{phase}/` folder)_"))
                continue
            imgs = sorted(p for p in phase_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
            if not imgs:
                display(Markdown(f"_(no images in `{phase}/`)_"))
                continue
            sampled = random.sample(imgs, min(N_PER_PHASE, len(imgs)))
            display(Markdown(f"### {phase} — {len(sampled)} of {len(imgs)} shown"))
            for p in sampled:
                display(Markdown(f"**{p.name}**  ({p.stat().st_size / 1024:.1f} KB)"))
                display(Image(filename=str(p)))

## 4. Cleanup: delete all generated images

Wipes everything under `outputs/generated_images/` (per-concept folders included). Use this between runs to keep the PVC tidy. Run only when you no longer need the previous run's artifacts.

In [ ]:
import shutil
from pathlib import Path

IMG_ROOT = Path("/scratch/Mock_repo_mnlp/outputs/generated_images")

if IMG_ROOT.exists():
    n_files = sum(1 for p in IMG_ROOT.rglob("*") if p.is_file())
    shutil.rmtree(IMG_ROOT)
    IMG_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Deleted {n_files} file(s) under {IMG_ROOT}")
else:
    print(f"Nothing to delete — {IMG_ROOT} does not exist")